# IMPORT LIBRARIES & LOAD DATASET

In [2]:
import pandas as pd
import numpy as np
from sklearn import datasets, linear_model
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [4]:
womens_world_cup_stats = pd.read_csv(
    '/Users/gui/womens_world_cup_2023_statsbomb_finalfinal.csv'
)

# FILTER SEMI-FINALS

In [9]:
semi_finals = womens_world_cup_stats[
    womens_world_cup_stats["stage"] == "Semi-finals"
].copy()

semi_finals.shape

(4, 53)

In [11]:
semi_finals.columns.tolist()

['match_id',
 'team',
 'match_date',
 'opponent',
 'stage',
 'goals_for',
 'goals_against',
 'result',
 'points',
 'possession_pct',
 'passes',
 'pass_accuracy_pct',
 'shots',
 'shots_on_target',
 'shot_accuracy_pct',
 'xG',
 'xG_per_shot',
 'goals_minus_xG',
 'conversion_pct',
 'big_chances',
 'key_passes',
 'corners',
 'crosses',
 'progressive_passes',
 'progressive_carries',
 'final_third_entries',
 'box_entries',
 'high_turnovers',
 'shots_after_turnover',
 'goals_after_turnover',
 'set_piece_goals',
 'tackles',
 'tackles_won',
 'interceptions',
 'blocks',
 'clearances',
 'recoveries',
 'saves',
 'clean_sheet',
 'formation',
 'shot_assists',
 'through_balls',
 'cutbacks',
 'dribbles',
 'successful_dribbles',
 'dribble_success_pct',
 'duels',
 'duels_won',
 'counterpress_actions',
 'goals_from_shots',
 'own_goals_for',
 'own_goals_against',
 'goal_type']

In [18]:
def min_max_score(series):
    if series.max() == series.min():
        return 50
    return ((series - series.min()) / (series.max() - series.min())) * 100

# CREATE THE SCORE OF THE 5 METRICS

In [20]:
semi_finals["xG_score"] = min_max_score(
    semi_finals["xG"]
)

semi_finals["pass_accuracy_score"] = min_max_score(
    semi_finals["pass_accuracy_pct"]
)

semi_finals["possession_score"] = min_max_score(
    semi_finals["possession_pct"]
)

semi_finals["goals_scored_score"] = min_max_score(
    semi_finals["goals_for"]
)

semi_finals["goals_conceded_score"] = (
    100 - min_max_score(semi_finals["goals_against"])
)

# PERFORMANCE SCORE

In [23]:
semi_finals["performance_score"] = (
    semi_finals["xG_score"] * 0.25 +
    semi_finals["pass_accuracy_score"] * 0.25 +
    semi_finals["possession_score"] * 0.10 +
    semi_finals["goals_scored_score"] * 0.25 +
    semi_finals["goals_conceded_score"] * 0.15
)

In [25]:
semi_finals["performance_score"] = semi_finals["performance_score"].round(1)

# FINAL TABLE

In [28]:
performance_table = semi_finals[
    [
        "team",
        "xG",
        "pass_accuracy_pct",
        "possession_pct",
        "goals_for",
        "goals_against",
        "performance_score"
    ]
].copy()



performance_table = performance_table.sort_values(
    "performance_score",
    ascending=False
).reset_index(drop=True)



performance_table.insert(
    0,
    "Rank",
    range(1, len(performance_table) + 1)
)



performance_table["xG"] = (
    performance_table["xG"].round(1)
)

performance_table["pass_accuracy_pct"] = (
    performance_table["pass_accuracy_pct"].round(1)
)

performance_table["possession_pct"] = (
    performance_table["possession_pct"].round(1)
)



performance_table = performance_table.rename(columns={
    "team": "Team",
    "xG": "xG (%)",
    "pass_accuracy_pct": "Pass Accuracy (%)",
    "possession_pct": "Possession (%)",
    "goals_for": "Goals Scored",
    "goals_against": "Goals Conceded",
    "performance_score": "Performance Score"
})



performance_table

,Rank,Team,xG (%),Pass Accuracy (%),Possession (%),Goals Scored,Goals Conceded,Performance Score
0,1,England Women's,1.1,83.2,50.0,3,1,82.4
1,2,Spain Women's,1.4,78.7,57.1,2,1,81.9
2,3,Australia Women's,1.1,71.7,50.0,1,3,31.7
3,4,Sweden Women's,0.7,63.1,42.9,1,2,7.5


## Table 2: Score Breakdown

In [31]:
# ==========================================
# SCORE BREAKDOWN
# ==========================================

score_breakdown = semi_finals[
    [
        "team",
        "xG_score",
        "pass_accuracy_score",
        "possession_score",
        "goals_scored_score",
        "goals_conceded_score",
        "performance_score"
    ]
].copy()



score_breakdown = score_breakdown.sort_values(
    "performance_score",
    ascending=False
).reset_index(drop=True)



score_breakdown.insert(
    0,
    "Rank",
    range(1, len(score_breakdown) + 1)
)



score_columns = [
    "xG_score",
    "pass_accuracy_score",
    "possession_score",
    "goals_scored_score",
    "goals_conceded_score",
    "performance_score"
]

score_breakdown[score_columns] = (
    score_breakdown[score_columns].round(1)
)



score_breakdown = score_breakdown.rename(columns={
    "team": "Team",
    "xG": "xG (25%)",
    "pass_accuracy_score": "Pass Accuracy Score (25%)",
    "possession_score": "Possession Score (10%)",
    "goals_scored_score": "Goals Scored Score (25%)",
    "goals_conceded_score": "Goals Conceded Score (15%)",
    "performance_score": "Performance Score"
})



score_breakdown

,Rank,Team,xG_score,Pass Accuracy Score (25%),Possession Score (10%),Goals Scored Score (25%),Goals Conceded Score (15%),Performance Score
0,1,England Women's,49.8,100.0,49.8,100.0,100.0,82.4
1,2,Spain Women's,100.0,77.8,100.0,50.0,100.0,81.9
2,3,Australia Women's,63.8,42.9,50.2,0.0,0.0,31.7
3,4,Sweden Women's,0.0,0.0,0.0,0.0,50.0,7.5


# Performance Score — Key Findings

England achieved the highest Performance Score (82.4), narrowly ahead of Spain (81.9). Despite Spain recording higher xG (1.4) and possession (57.1%), England benefited from a much higher pass accuracy (83.2%) and a stronger attacking outcome, scoring three goals from 1.1 xG and conceding only one. This highlights how efficiency and results can outweigh greater possession and chance creation in the overall performance score.

Australia ranked third despite recording similar possession and xG to England, mainly because they failed to convert their chances and conceded three goals. Sweden had the lowest score (7.5), reflecting weaker values across most metrics, particularly xG, passing accuracy, possession and goals scored. Overall, the ranking shows that the best-performing team was not necessarily the one with the highest possession or xG, but the one that combined efficiency with a positive result.